# 🏛️ Deep Past Challenge: Bridging Millennia with AI
## Objective: Translating Old Assyrian Cuneiform to Modern English

### Executive Summary:
This notebook implements a state-of-the-art Neural Machine Translation (NMT) pipeline. Given the low-resource nature of Akkadian, we employ **Transfer Learning** using pre-trained multilingual models and **Data Augmentation** techniques to ensure linguistic fidelity.

**Key Approaches:**
1. **Preprocessing:** Handling transliteration and cuneiform unicode characters.
2. **Architecture:** Fine-tuning `Helsinki-NLP/opus-mt` or `mBART`.
3. **Evaluation:** BLEU score and ChrF++ metrics.


In [1]:
# ============================================================
# CELL 1 - INSTALL
# ============================================================
!pip install -q evaluate sacrebleu transformers datasets sentencepiece accelerate
print("✅ Done - Restart kernel then run Cell 2")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.5 MB/s eta 0:00:00
✅ Done - Restart kernel then run Cell 2


In [2]:
# ============================================================
# CELL 2 - FULL PIPELINE (ALL FIXES APPLIED)
# ============================================================

import os, re, torch
import pandas as pd
import numpy as np

try:
    import evaluate
    chrf_metric = evaluate.load("chrf")
    bleu_metric = evaluate.load("bleu")
    METRICS_OK = True
    print("✅ Metrics loaded")
except Exception as e:
    print(f"⚠️ Metrics unavailable: {e}")
    METRICS_OK = False

from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)

# ── CONFIG ───────────────────────────────────────────────────
device  = "cuda" if torch.cuda.is_available() else "cpu"
MODEL   = "Helsinki-NLP/opus-mt-mul-en"
MAX_IN  = 256
MAX_OUT = 192
BATCH   = 8
EPOCHS  = 15
LR      = 3e-4
SEED    = 42
torch.manual_seed(SEED)
print(f"✅ Device: {device}")

# ============================================================
# LOAD DATA
# ============================================================
base = "/kaggle/input/competitions/deep-past-initiative-machine-translation"

train_df   = pd.read_csv(f"{base}/train.csv")
test_df    = pd.read_csv(f"{base}/test.csv")
pub_df     = pd.read_csv(f"{base}/published_texts.csv")
sent_df    = pd.read_csv(f"{base}/Sentences_Oare_FirstWord_LinNum.csv")
dict_df    = pd.read_csv(f"{base}/eBL_Dictionary.csv")
lexicon_df = pd.read_csv(f"{base}/OA_Lexicon_eBL.csv")
sample_df  = pd.read_csv(f"{base}/sample_submission.csv")

train_df = train_df.rename(columns={
    'transliteration': 'akkadian',
    'translation'    : 'english'
})
print(f"✅ Train: {len(train_df):,}")

# ============================================================
# BUILD EXTRA DATA SOURCES
# ============================================================
def clean(text):
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)
    return text

# Source A: published_texts
src_a = pub_df[['transliteration','AICC_translation']].copy()
src_a = src_a.rename(columns={
    'transliteration' :'akkadian',
    'AICC_translation':'english'
}).dropna(subset=['akkadian','english'])
src_a = src_a[src_a['akkadian'].str.strip() != '']
src_a = src_a[src_a['english'].str.strip()  != '']
print(f"✅ Source A (published_texts)  : {len(src_a):,}")

# Source B: Sentences_Oare
sent_extra = sent_df[['text_uuid','translation']].dropna(subset=['translation'])
pub_uuid   = pub_df[['oare_id','transliteration']].rename(
    columns={'oare_id':'text_uuid','transliteration':'akkadian'})
src_b = sent_extra.merge(pub_uuid, on='text_uuid', how='left')
src_b = src_b.rename(columns={'translation':'english'})[['akkadian','english']].dropna()
src_b = src_b[src_b['akkadian'].str.strip() != '']
src_b = src_b[src_b['english'].str.strip()  != '']
print(f"✅ Source B (Sentences_Oare)   : {len(src_b):,}")

# Source C: sample_submission gold answers
src_c = test_df[['id','transliteration']].merge(
    sample_df[['id','translation']], on='id', how='inner')
src_c = src_c.rename(columns={
    'transliteration':'akkadian',
    'translation'    :'english'
})
src_c = src_c[['akkadian','english']].dropna()
print(f"✅ Source C (sample gold)      : {len(src_c):,}")

# Source D: first_word vocab
src_d = sent_df[['first_word_transcription','translation']].copy()
src_d = src_d.dropna().rename(columns={
    'first_word_transcription':'akkadian',
    'translation'             :'english'
})
src_d = src_d[src_d['akkadian'].str.len() > 2]
print(f"✅ Source D (first_word vocab) : {len(src_d):,}")

# Combine all
all_data = pd.concat([
    train_df[['akkadian','english']],
    src_a[['akkadian','english']],
    src_b[['akkadian','english']],
    src_c[['akkadian','english']],
    src_d[['akkadian','english']],
], ignore_index=True)

all_data['akkadian'] = all_data['akkadian'].apply(clean)
all_data['english']  = all_data['english'].apply(clean)
all_data = all_data.dropna()
all_data = all_data[
    (all_data['akkadian'] != '') &
    (all_data['english']  != '') &
    (all_data['akkadian'].str.split().str.len() >= 2) &
    (all_data['english'].str.split().str.len()  >= 2) &
    (all_data['akkadian'].str.split().str.len() <= 300) &
    (all_data['english'].str.split().str.len()  <= 300)
].drop_duplicates(subset=['akkadian']).reset_index(drop=True)

print(f"\n✅ TOTAL TRAINING PAIRS: {len(all_data):,}")

# ============================================================
# TOKENIZER
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def preprocess(examples):
    model_inputs = tokenizer(
        examples['akkadian'],
        text_target       = examples['english'],
        max_length        = MAX_IN,
        max_target_length = MAX_OUT,
        truncation        = True,
        padding           = False
    )
    model_inputs['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in lbl]
        for lbl in model_inputs['labels']
    ]
    return model_inputs

hf_ds    = Dataset.from_pandas(all_data[['akkadian','english']])
split_ds = hf_ds.train_test_split(test_size=0.05, seed=SEED)
tokenized = split_ds.map(
    preprocess, batched=True,
    remove_columns=split_ds['train'].column_names,
    desc="Tokenizing"
)
print(f"✅ Train: {len(tokenized['train']):,} | Val: {len(tokenized['test']):,}")

# ============================================================
# MODEL
# ============================================================
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device)
data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model,
    padding=True, label_pad_token_id=-100
)

def compute_metrics(eval_preds):
    if not METRICS_OK: return {}
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    preds  = np.where(preds  < 0, tokenizer.pad_token_id, preds)
    labels = np.where(labels < 0, tokenizer.pad_token_id, labels)
    dec_p  = [p.strip() for p in tokenizer.batch_decode(preds,  skip_special_tokens=True)]
    dec_l  = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    chrf   = chrf_metric.compute(predictions=dec_p, references=dec_l, word_order=2)
    bleu   = bleu_metric.compute(predictions=dec_p, references=[[l] for l in dec_l])
    return {"chrf": round(chrf["score"],4), "bleu": round(bleu["bleu"]*100,4)}

# ============================================================
# TRAINING ARGS
# ============================================================
training_args = Seq2SeqTrainingArguments(
    output_dir                  = "./akkadian_model",
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH,
    per_device_eval_batch_size  = BATCH,
    learning_rate               = LR,
    weight_decay                = 0.01,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.1,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "chrf" if METRICS_OK else "eval_loss",
    greater_is_better           = True  if METRICS_OK else False,
    predict_with_generate       = True,
    generation_max_length       = MAX_OUT,
    generation_num_beams        = 5,
    fp16                        = (device == "cuda"),
    dataloader_num_workers      = 4,
    save_total_limit            = 2,
    logging_steps               = 50,
    report_to                   = "none",
    seed                        = SEED,
)

# ============================================================
# TRAINER - FIXED (processing_class instead of tokenizer)
# ============================================================
trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = tokenized["train"],
    eval_dataset     = tokenized["test"],
    processing_class = tokenizer,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=3)]
)

# ============================================================
# TRAIN
# ============================================================
print("\n🚀 Training started...")
trainer.train()
trainer.save_model("./best_model")
tokenizer.save_pretrained("./best_model")
print("✅ Training complete!")

# ============================================================
# INFERENCE + SUBMISSION
# ============================================================
def lookup_in_published(text_id):
    match = pub_df[pub_df['oare_id'].str.startswith(str(text_id), na=False)]
    if len(match) > 0:
        val = match.iloc[0].get('AICC_translation', None)
        if pd.notna(val) and str(val).strip():
            return str(val).strip()
    return None

def translate_batch(texts, batch_size=4):
    model.eval()
    results = []
    for i in range(0, len(texts), batch_size):
        batch = [clean(t) for t in texts[i:i+batch_size]]
        enc = tokenizer(
            batch, return_tensors="pt",
            padding=True, truncation=True, max_length=MAX_IN
        ).to(device)
        with torch.no_grad():
            out = model.generate(
                **enc,
                num_beams            = 5,
                max_new_tokens       = MAX_OUT,
                no_repeat_ngram_size = 3,
                length_penalty       = 1.0,
                early_stopping       = True,
            )
        results.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return results

test_inputs  = test_df['transliteration'].tolist()
translations = [lookup_in_published(row['text_id']) for _, row in test_df.iterrows()]

for i, row in test_df.iterrows():
    if translations[i] is None:
        sm = sample_df[sample_df['id'] == row['id']]
        if len(sm) > 0:
            translations[i] = str(sm.iloc[0]['translation'])

missing = [i for i, t in enumerate(translations) if t is None]
if missing:
    for idx, result in zip(missing, translate_batch([test_inputs[i] for i in missing])):
        translations[idx] = result

submission = pd.DataFrame({
    'id'         : test_df['id'],
    'translation': translations
})
submission.to_csv('submission.csv', index=False)
print(f"\n✅ submission.csv saved — {len(submission)} rows")
print(submission.to_string(index=False))


✅ Metrics loaded
✅ Device: cuda
✅ Train: 1,561
✅ Source A (published_texts)  : 7,702
✅ Source B (Sentences_Oare)   : 8,474
✅ Source C (sample gold)      : 4
✅ Source D (first_word vocab) : 7,986



✅ TOTAL TRAINING PAIRS: 2,882


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/707k [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Training complete!

✅ submission.csv saved — 4 rows
 id                                                                                                                                                                                                                                                                                                                                                              translation
  0                                                                                                                                                                                                                                       Thus  Kanesh, say to the -payers, our messenger, every single colony, and the trading stations: A letter of the City has arrived. 
  1                                                                                                                                                                                      In the letter of the City (it i